# 🚀 Treinamento de IA de Mangá (Qwen 2.5 Manga PT-BR) com Unsloth no Google Colab
### Crie o seu próprio modelo de IA especializado em tradução de mangás e quadrinhos!
Este notebook utiliza **Unsloth** para realizar Fine-Tuning (LoRA / QLoRA) no **Qwen 2.5** utilizando o seu próprio histórico de traduções (`dataset_manga_ptbr.jsonl`).

**Vantagens do modelo próprio:**
* **Custo Zero:** Treina e roda de graça na GPU T4 do Google Colab ou na GPU do seu computador.
* **Sem Limite de Tokens (Zero Bloqueio TPM/TPD):** Não gasta chaves de API do Groq ou OpenRouter.
* **Sem Censura e com Vocabulário Perfeito:** Aprende de nascença gírias eróticas (+18), oralidade brasileira e nomes de personagens sem precisar de prompts gigantescos.
* **Exportação Automática para Ollama:** Gera o arquivo `.gguf` pronto para rodar no Manga Studio com 1 clique.

---
### ⚡ Pré-requisito: Ative a GPU T4
Vá no menu superior em **Ambiente de execução** > **Alterar tipo de ambiente de execução** e escolha **GPU T4**.

In [ ]:
# @title 1. Conectar Google Drive (Local do seu Dataset)
from google.colab import drive
import os, shutil

drive.mount("/content/drive")
dataset_candidates = [
    "/content/drive/MyDrive/MANGAS [PT-BR]/dataset_manga_ptbr.jsonl",
    "/content/drive/MyDrive/dataset_manga_ptbr.jsonl",
    "/content/dataset_manga_ptbr.jsonl"
]

dataset_file = None
for p in dataset_candidates:
    if os.path.exists(p):
        dataset_file = p
        break

if dataset_file:
    print(f"✅ Dataset encontrado: {dataset_file}")
else:
    print("⚠️ Dataset não encontrado no Drive. Por favor, exporte `dataset_manga_ptbr.jsonl` pelo Manga Studio.")


In [ ]:
# @title 2. Instalar Unsloth e Bibliotecas de Fine-Tuning
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
print("✅ Unsloth instalado com sucesso!")


In [ ]:
# @title 3. Configurar e Carregar o Modelo Base (Qwen 2.5 4-bit)
tamanho_modelo = "7b" # @param ["3b", "7b", "14b"]
model_name = f"unsloth/Qwen2.5-{tamanho_modelo}-Instruct-bnb-4bit"

from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

print(f"Carregando {model_name} com aceleração Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("✅ Modelo base pronto com adaptadores LoRA!")


In [ ]:
# @title 4. Preparar Dataset para Treino
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"},
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

raw_dataset = load_dataset("json", data_files=dataset_file, split="train")
dataset = raw_dataset.map(formatting_prompts_func, batched=True)
print(f"✅ Dataset pronto! {len(dataset)} exemplos prontos para treinamento.")


In [ ]:
# @title 5. Iniciar Treinamento do Modelo Especializado
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

epocas = 2 # @param {type:"slider", min:1, max:5, step:1}
batch_size = 2 # @param {type:"slider", min:1, max:4, step:1}

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=epocas,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs_manga",
    ),
)

print("🚀 Treinamento iniciado! Isso levará cerca de 5 a 15 minutos dependendo do tamanho...")
trainer_stats = trainer.train()
print("🎉 Treinamento concluído com sucesso!")


In [ ]:
# @title 6. Exportar Modelo para GGUF (Compatível com Ollama / Manga Studio)
save_dir = "/content/drive/MyDrive/.ollama_models"
os.makedirs(save_dir, exist_ok=True)

print("Exportando modelo para formato quantizado GGUF (Q4_K_M)...")
model.save_pretrained_gguf(
    "qwen-manga-ptbr",
    tokenizer,
    quantization_method="q4_k_m"
)

# Move o arquivo GGUF para o Google Drive para ser permanente
!cp qwen-manga-ptbr/qwen-manga-ptbr-unsloth.Q4_K_M.gguf "/content/drive/MyDrive/qwen-manga-ptbr.Q4_K_M.gguf" 2>/dev/null || true

# Cria o Modelfile padrão para o Ollama
modelfile_content = """FROM ./qwen-manga-ptbr.Q4_K_M.gguf
PARAMETER temperature 0.2
PARAMETER stop "<|im_end|>"
SYSTEM """Você é o tradutor literário oficial de scanlation de mangás para PT-BR. Adapte com oralidade coloquial autêntica, termos sem censura e naturalidade."""
"""
with open("/content/drive/MyDrive/Modelfile.qwen-manga", "w", encoding="utf-8") as mf:
    mf.write(modelfile_content)

print("✅ Modelo GGUF e Modelfile salvos com sucesso no seu Google Drive!")
print("👉 Arquivo: Google Drive > qwen-manga-ptbr.Q4_K_M.gguf")


In [ ]:
# @title 7. (Opcional) Testar Inferência da IA Treinada Aqui Mesmo
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Você é o tradutor oficial de mangá para PT-BR."},
    {"role": "user", "content": "Traduza o seguinte diálogo: \"LOOK LIKE MY LITTLE BEAN CAN'T HOLD BACK ANYMORE! HE HE~\""}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True)
print("\n--- RESULTADO DA IA TREINADA ---")
print(tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0])
